In [2]:
import pandas as pd
import plotly.express as px

df = pd.read_excel('Sales_Lead_Dataset.xlsx')
df['Lead Create Date'] = pd.to_datetime(df['Lead Create Date'])

In [9]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
# total number of leads
total_leads = len(df)
print(f"Total Leads: {total_leads}")

# breakdown of lead statuses
status_counts = df['Lead Status'].value_counts()
print("\nLead Status Breakdown:")
print(status_counts)

# calculate enrollment rate
if 'Enrolled' in status_counts:
    enrolled_leads = status_counts['Enrolled']
    enrollment_rate = (enrolled_leads / total_leads) * 100
    print(f"\nOverall Enrollment Rate: {enrollment_rate:.1f}%")


Total Leads: 401

Lead Status Breakdown:
Lead Status
Enrolled       292
No Response     70
Connecting      23
Lost            10
Expired          5
Canceled         1
Name: count, dtype: int64

Overall Enrollment Rate: 72.8%


In [5]:
#  how many leads were connected to a group, and how well those specific leads converted compared to individual leads
group_impact = pd.crosstab(df['Group Connected'], df['Lead Status'], margins=True)
print("Lead Status by Group Connection:")
print(group_impact)


Lead Status by Group Connection:
Lead Status      Canceled  Connecting  Enrolled  Expired  Lost  No Response  \
Group Connected                                                               
No                      0          22       100        5     2           59   
Yes                     1           1       192        0     8           11   
All                     1          23       292        5    10           70   

Lead Status      All  
Group Connected       
No               188  
Yes              213  
All              401  


In [6]:
# average follow-up count for each lead
avg_follow_ups = df.groupby('Lead Status')['Follow Up Count'].mean().round(2).sort_values()
print("Average Follow-Ups per Lead Status:")
print(avg_follow_ups)

Average Follow-Ups per Lead Status:
Lead Status
Enrolled       1.55
Connecting     2.52
Canceled       3.00
Expired        3.20
Lost           3.70
No Response    5.23
Name: Follow Up Count, dtype: float64


In [7]:
student_conversion = pd.crosstab(df['Student Category'], df['Lead Status'])
print("Lead Status by Student Category:")
print(student_conversion)

Lead Status by Student Category:
Lead Status       Canceled  Connecting  Enrolled  Expired  Lost  No Response
Student Category                                                            
Graduate                 0          10       105        1     4           19
J1                       0           3        28        0     2            6
OPT                      1           4        63        3     1           11
Undergraduate            0           6        96        1     3           34


# Overall Executive Summary

Based on the calculations above and the graphs show in following dashbroad, the current lead pipeline demonstrates strong overall performance with an impressive **72.8% enrollment rate** (292 enrolled out of 401 total leads). However, deeper analysis reveals clear opportunities to optimize sales resources and improve conversion rates for underperforming segments.

### Key Insights & Action

*   **Group Connections Drive Conversions:** Leads connected to a group enroll at a significantly higher rate. Out of 213 group-connected leads, 192 enrolled (a 90.1% success rate). By contrast, leads without a group connection make up the majority of "No Response" leads (59 out of 70) and convert at only 53.2%.
    *   *Action:* Marketing should prioritize campaigns that generate group-affiliated leads, as they are highly valuable and convert faster.

*   **Cap Manual Follow-Ups to Save Time:** High-intent leads enroll quickly, requiring an average of only 1.55 follow-ups. In contrast, "No Response" leads consume significant time, receiving an average of 5.23 follow-ups.
    *   *Action:* The sales team should cap manual outreach at 3 to 4 attempts. Unresponsive leads should then be moved to an automated email sequence to maximize sales efficiency.

*   **Adjust Messaging for Undergraduates:** Graduate students are our most successful segment (105 enrollments). However, Undergraduate students are the most likely to disengage, making up nearly half of all "No Response" leads (34 out of 70).
    *   *Action:*Marketing should rethink how we talk to Undergraduates, specifically for Undergraduates. They may need more introductory information or different incentives compared to Graduate or OPT students.

*   **Re-Engage Stalled Leads:** Despite a strong overall conversion rate, 93 leads are currently stalled in "No Response" (70) or "Connecting" (23) statuses.
    *   *Action:* Launch a targeted re-engagement campaign featuring a limited-time offer to motivate these pending leads to complete their enrollment.



In [8]:
# Lead Status Distribution
status_counts = df['Lead Status'].value_counts().reset_index()
status_counts.columns = ['Lead Status', 'Count']

fig1 = px.bar(status_counts,
              x='Lead Status',
              y='Count',
              color='Lead Status',
              title='Total Leads by Current Status',
              text='Count')

fig1.update_traces(textposition='outside')
fig1.show()


This chart displays the overall health of the sales pipeline by breaking down the total number of leads in each status category.
*   **Key Insight:** The pipeline is highly successful, with the vast majority of leads successfully reaching the "Enrolled" status (292). However, the primary bottleneck is the "No Response" category (70 leads), which represents the largest part of unconverted opportunities.
*   **Action:** Deploy an automated re-engagement campaign (such as a limited-time offer or automated email sequence) targeting the "No Response" segment to recover stalled leads without burning manual sales hours.


In [ ]:
# understand which student demographics convert to "Enrolled" best
cat_status = df.groupby(['Student Category', 'Lead Status']).size().reset_index(name='Count')

fig2 = px.bar(cat_status,
              x='Student Category',
              y='Count',
              color='Lead Status',
              title='Lead Status Distribution by Student Category',
              barmode='stack')

fig2.show()

This stacked bar chart breaks down lead outcomes (such as Enrolled, Connecting, or No Response) across four distinct student demographic groups (Graduate, J1, OPT, Undergraduate).
*   **Key Insight:** Graduate and Undergraduate students make up the large of our total leads. However, while Graduate students show a very strong and efficient conversion to "Enrolled," Undergraduate students have a disproportionately large "No Response" segment (the orange block).
*   **Action:** Adjust the communication strategy specifically for Undergraduates. Marketing should test new messaging, offer different early-stage incentives, or provide more accessible upfront information to prevent this demographic from disengaging.


In [ ]:
#the relationship between the number of follow-ups and the final lead status
fig3 = px.box(df,
              x='Lead Status',
              y='Follow Up Count',
              color='Lead Status',
              title='Follow-Up Count Distribution vs. Lead Status')

fig3.show()


This box plot illustrates the number of follow-up attempts made across different lead statuses, revealing how sales effort correlates with final outcomes.
*   **Key Insight:** High-intent leads convert almost immediately, with the vast majority of "Enrolled" leads requiring only a single follow-up. In stark contrast, the sales team is expending maximum effort (averaging 5 to 6 follow-ups) on "No Response" leads, resulting in a significant drain on time and resources with zero conversion.
*   **Action:** Implement a hard upper limits on sales follow-ups at 3 attempts. After the third attempt, automatically transition unresponsive leads into an automated, low-touch email nurture sequence to protect the sales team's bandwidth for high-converting prospects.


In [ ]:
# if being connected to a group makes more likely to enroll?

group_df = df.dropna(subset=['Group Connected'])
group_status = group_df.groupby(['Group Connected', 'Lead Status']).size().reset_index(name='Count')

fig4 = px.bar(group_status,
              x='Group Connected',
              y='Count',
              color='Lead Status',
              title='Impact of Group Connection on Lead Outcome',
              barmode='group')

fig4.show()

This grouped bar chart compares the final status of leads based on whether they successfully established a "Group Connection" (Yes vs. No) during the sales process.
*   **Key Insight:** Establishing a group connection is a massive driver of conversion. Leads that are connected to a group ("Yes") overwhelmingly convert to "Enrolled" with almost zero drop-off. Conversely, leads without a group connection ("No") account for almost all of the "No Response" and "Connecting" pipeline bottlenecks.
*   **Action:** Focus early sales efforts entirely on getting people connected to a group. We should redesign our initial outreach so that joining a group is the obvious, immediate next step for every new lead.

In [ ]:
#measuring marketing campaign spikes
time_df = df.groupby([df['Lead Create Date'].dt.to_period("W").astype(str), 'Lead Status']).size().reset_index(name='Count')
time_df.rename(columns={'Lead Create Date': 'Week'}, inplace=True)

fig5 = px.line(time_df,
               x='Week',
               y='Count',
               color='Lead Status',
               markers=True,
               title='Weekly Lead Generation by Status')

fig5.show()


This line graph tracks how many leads were generated week by week and what their final outcome was. It helps us see seasonal trends and how lead volume changes over time.
*   **Key Insight:** Enrollments aren't stable. It happen in big waves. There is a massive increasing in late August (likely a "back-to-school" rush or a major marketing campaign). Interestingly, during this huge rush, the number of "No Response" leads also increased. This suggests that the sales team might have been overwhelmed by the sudden volume and couldn't keep up with everyone.
*   **Action:** Prepare early for the late August rush. We need to schedule extra staff or set up automated welcome emails before the busy season hits, ensuring no leads fall through the cracks just because the team is too busy to follow up.
